In [1]:
from typing import List, TypedDict
import time

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate


/var/folders/sb/2kl68frj62d7gyw1rfz639l00000gn/T/ipykernel_46651/4114872307.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
docs = (
    PyPDFLoader("../documents/book1.pdf").load() +
    PyPDFLoader("../documents/book2.pdf").load() +
    PyPDFLoader("../documents/book3.pdf").load()
)

In [4]:
len(docs)

2123

In [5]:
# 2) Chunk
chunks = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=150).split_documents(docs)

# 3) Clean text to avoid UnicodeEncodeError (surrogates from PDF extraction)
for d in chunks:
    d.page_content = d.page_content.encode("utf-8", "ignore").decode("utf-8", "ignore")

In [6]:
len(chunks)

6399

In [8]:
# 3) Index (fresh collection each run)
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_message

embeddings = GoogleGenerativeAIEmbeddings(model='models/gemini-embedding-001')

# Free-tier quota is 100 embed_content requests/minute, so batch small and
# back off with retries when we get a 429 RESOURCE_EXHAUSTED.
EMBED_BATCH_SIZE = 90


@retry(
    retry=retry_if_exception_message(match=".*RESOURCE_EXHAUSTED.*"),
    wait=wait_exponential(multiplier=1, min=10, max=90),
    stop=stop_after_attempt(6),
)
def add_batch(store, batch):
    if store is None:
        return FAISS.from_documents(batch, embeddings)
    store.add_documents(batch)
    return store


vector_store = None
for i in range(0, len(chunks), EMBED_BATCH_SIZE):
    batch = chunks[i : i + EMBED_BATCH_SIZE]
    vector_store = add_batch(vector_store, batch)
    time.sleep(2)  # stay comfortably under the requests/minute quota

KeyboardInterrupt: 

In [ ]:
retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k':4})

NameError: name 'vector_store' is not defined